In [2]:
import pandas as pd
import numpy as np 


In [3]:
data_path='../data/raw/diabetic_data.csv'

df=pd.read_csv(data_path)

In [4]:
df.sample(3)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
65703,183371436,86813001,Caucasian,Female,[40-50),?,1,1,7,3,...,No,No,No,No,No,No,No,No,Yes,NO
4856,26411316,69532245,Caucasian,Female,[60-70),?,6,1,7,6,...,No,Steady,No,No,No,No,No,No,Yes,>30
77452,235146816,68243418,AfricanAmerican,Female,[80-90),?,1,3,7,2,...,No,Down,No,No,No,No,No,Ch,Yes,>30


In [6]:
df.shape # (101766,50) 

(101766, 50)

In [8]:
# finding Duplicates

df.duplicated().sum() # 0 Duplication 

np.int64(0)

In [21]:
null_values=((df.isnull().mean())*100).sort_values(ascending=False)
null_values=null_values[null_values>0]
null_values


# max_glu_serum    94.746772
# A1Cresult        83.277322

max_glu_serum    94.746772
A1Cresult        83.277322
dtype: float64

In [23]:
# now Analyzing the target Column Readmitted

target_analysis=df['readmitted'].value_counts()
target_analysis
#understanding The data
# NO     54864  - Not admitted 
# >30    35545   - Admitted after 30 days
# <30    11357   - Main Business problem - Admitted Before Time Period Ended 

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [30]:

target_summary = pd.DataFrame({
    'Count': df['readmitted'].value_counts(),
    'Percentage (%)': (df['readmitted'].value_counts(normalize=True) * 100).round(2)
})

# now I Will Apply the Lambda and Map To show Result in the Form of Df
target_summary['Binary Class'] = target_summary.index.map(lambda x: 'Class 1 (High Risk)' if x == '<30' else 'Class 0 (Safe/Late)')

target_summary

,Count,Percentage (%),Binary Class
readmitted,,,
NO,54864,53.91,Class 0 (Safe/Late)
>30,35545,34.93,Class 0 (Safe/Late)
<30,11357,11.16,Class 1 (High Risk)


In [ ]:
# mostly Columns nan values Filled With ? so Replacing Them with np.nan 
new_df=df.replace('?',np.nan)

In [32]:
null_values=((new_df.isnull().mean())*100).sort_values(ascending=False)
null_values=null_values[null_values>0]
null_values


weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64

In [35]:
# drop columns That are Not Useful 
columns_to_drop = ['weight', 'payer_code', 'encounter_id', 'patient_nbr']
new_df = new_df.drop(columns=columns_to_drop)

# missing  null values with logical data 
 
new_df['medical_specialty'] = new_df['medical_specialty'].fillna('Missing')
new_df['max_glu_serum'] = new_df['max_glu_serum'].fillna('Not Tested')
new_df['A1Cresult'] = new_df['A1Cresult'].fillna('Not Tested')

# here We Convert Column In to Binary 
new_df['target'] = (new_df['readmitted'] == '<30').astype(int)
new_df = new_df.drop(columns=['readmitted'])

In [38]:
print("Final Cleaned Shape:", new_df.shape)
print("Target Balance:\n", new_df['target'].value_counts(normalize=True) * 100)


Final Cleaned Shape: (101766, 46)
Target Balance:
 target
0    88.840084
1    11.159916
Name: proportion, dtype: float64


In [39]:
new_df.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'medical_specialty', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
       'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'target'],
      dtype='str')

In [51]:
print(new_df[['examide','citoglipton']].value_counts()) #No       No             101766

# these 2 columns are not useful because both giving 100% No so Its A waste of Memory 

examide  citoglipton
No       No             101766
Name: count, dtype: int64


In [ ]:

new_df = new_df.drop(columns=['examide', 'citoglipton'])


expired_ids = [11, 13, 14, 19, 20, 21]


new_df = new_df[~new_df['discharge_disposition_id'].isin(expired_ids)]


print("Shape after filtering expired patients:", new_df.shape) # 99343 , 44

Shape after filtering expired patients: (99343, 44)


In [55]:
new_df.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'medical_specialty', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
       'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'target'],
      dtype='str')

In [ ]:
def map_icd9_to_category(code):
    if pd.isna(code) or code == 'Missing':
        return 'Missing'
    
    
    code_str = str(code).strip()
    
    
    if code_str.startswith('V') or code_str.startswith('E'):
        return 'Other'
    
    try:
        val = float(code_str)
    except ValueError:
        return 'Other'
        
    
    if (390 <= val <= 459) or val == 785:
        return 'Circulatory'
    elif (460 <= val <= 519) or val == 786:
        return 'Respiratory'
    elif (520 <= val <= 579) or val == 787:
        return 'Digestive'
    elif 250 <= val < 251:
        return 'Diabetes'
    elif 800 <= val <= 999:
        return 'Injury'
    elif 710 <= val <= 739:
        return 'Musculoskeletal'
    elif (580 <= val <= 629) or val == 788:
        return 'Genitourinary'
    elif 140 <= val <= 239:
        return 'Neoplasms'
    else:
        return 'Other'


print(map_icd9_to_category('250.01')) # diabetes
print(map_icd9_to_category('414'))     # Circulatory

# we change the Code in To a proper Disease name 

Diabetes
Circulatory


In [57]:

for col in ['diag_1', 'diag_2', 'diag_3']:
    new_df[col] = new_df[col].apply(map_icd9_to_category)


print(new_df['diag_1'].value_counts())

diag_1
Circulatory        29681
Other              17793
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6853
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
Missing               20
Name: count, dtype: int64


In [58]:
from sklearn.model_selection import train_test_split

X = new_df.drop(columns=['target'])
y = new_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")
print(f"Train High Risk Ratio: {y_train.mean():.4f}")
print(f"Test High Risk Ratio:  {y_test.mean():.4f}")

X_train Shape: (79474, 43)
X_test Shape:  (19869, 43)
Train High Risk Ratio: 0.1139
Test High Risk Ratio:  0.1139


In [59]:

num_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
    'num_medications', 'number_outpatient', 'number_emergency', 
    'number_inpatient', 'number_diagnoses'
]

cat_cols = [col for col in X_train.columns if col not in num_cols]

print(f"Numerical Columns Count:   {len(num_cols)}")
print(f"Categorical Columns Count: {len(cat_cols)}")
print("Total Features:", len(num_cols) + len(cat_cols))

Numerical Columns Count:   8
Categorical Columns Count: 35
Total Features: 43


In [62]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. ColumnTransformer (Combining both sub-pipelines)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ]
)

In [63]:

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train Shape:", X_train_processed.shape)
print("Processed X_test Shape: ", X_test_processed.shape)

Processed X_train Shape: (79474, 255)
Processed X_test Shape:  (19869, 255)


In [65]:

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score



imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()

#  Random Forest Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

#  XGBoost Pipeline
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        scale_pos_weight=imbalance_ratio, # Handling imbalance
        random_state=42, 
        n_jobs=-1
    ))
])

# Random Forest Train & Predict
rf_pipeline.fit(X_train, y_train)
rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]

# XGBoost Train & Predict
xgb_pipeline.fit(X_train, y_train)
xgb_probs = xgb_pipeline.predict_proba(X_test)[:, 1]

# custom Reusable Function to check performance 
def evaluate_model(name, y_true, y_probs):
    roc_auc = roc_auc_score(y_true, y_probs)
    pr_auc = average_precision_score(y_true, y_probs)
    print(f"{name} Performance ")
    print(f"ROC-AUC Score: {roc_auc:.4f}")
    print(f"PR-AUC Score:  {pr_auc:.4f}")
    print("-" * 30)

evaluate_model("Random Forest Baseline", y_test, rf_probs)
evaluate_model("XGBoost Baseline", y_test, xgb_probs)

Random Forest Baseline Performance 
ROC-AUC Score: 0.6500
PR-AUC Score:  0.2184
------------------------------
XGBoost Baseline Performance 
ROC-AUC Score: 0.6755
PR-AUC Score:  0.2333
------------------------------


In [ ]:

feature_names = xgb_pipeline.named_steps['preprocessor'].get_feature_names_out()
xgb_model = xgb_pipeline.named_steps['classifier']

importances_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)


importances_df.head(10)

,Feature,Importance
35,cat__discharge_disposition_id_1,0.062643
6,num__number_inpatient,0.059845
50,cat__discharge_disposition_id_22,0.020160
150,cat__diag_1_Musculoskeletal,0.016550
39,cat__discharge_disposition_id_5,0.011066
36,cat__discharge_disposition_id_2,0.009676
32,cat__admission_type_id_6,0.009586
53,cat__discharge_disposition_id_25,0.009531
40,cat__discharge_disposition_id_6,0.009271
55,cat__discharge_disposition_id_28,0.009017


In [67]:
# tunning The data or model 

from sklearn.model_selection import RandomizedSearchCV


param_grid = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.03, 0.05, 0.1],
    'classifier__subsample': [0.7, 0.8, 0.9],
    'classifier__colsample_bytree': [0.6, 0.8, 1.0],
    'classifier__scale_pos_weight': [imbalance_ratio]  # Class imbalance scaling
}


search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_grid,
    n_iter=10,             
    scoring='roc_auc',
    cv=3,                  
    random_state=42,
    n_jobs=-1,
    verbose=1
)


print("Tuning XGBoost Model Parameters...")
search.fit(X_train, y_train)


best_xgb_pipeline = search.best_estimator_

print("\nBest Parameters Found:")
print(search.best_params_)
print(f"\nBest Cross-Validation ROC-AUC: {search.best_score_:.4f}")

Tuning XGBoost Model Parameters...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Parameters Found:
{'classifier__subsample': 0.7, 'classifier__scale_pos_weight': np.float64(7.780687216882113), 'classifier__n_estimators': 200, 'classifier__max_depth': 7, 'classifier__learning_rate': 0.03, 'classifier__colsample_bytree': 0.6}

Best Cross-Validation ROC-AUC: 0.6689


In [ ]:

def add_domain_features(df):
    df = df.copy()

    df['service_utilization'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
    

    df['med_per_day'] = df['num_medications'] / (df['time_in_hospital'] + 1e-5)
    
   
    df['lab_per_day'] = df['num_lab_procedures'] / (df['time_in_hospital'] + 1e-5)
    
  
    df['health_index'] = df['service_utilization'] * df['number_diagnoses']
    
    return df


X_train_fe = add_domain_features(X_train)
X_test_fe = add_domain_features(X_test)


num_cols_fe = num_cols + ['service_utilization', 'med_per_day', 'lab_per_day', 'health_index']

print("New Numerical Features Count:", len(num_cols_fe))
print("Updated X_train Shape:", X_train_fe.shape)

New Numerical Features Count: 12
Updated X_train Shape: (79474, 47)


In [ ]:

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols_fe),
        ('cat', cat_pipeline, cat_cols)
    ]
)

best_params = {k.replace('classifier__', ''): v for k, v in search.best_params_.items()}


final_xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier', XGBClassifier(**best_params, random_state=42, n_jobs=-1))
])


final_xgb_pipeline.fit(X_train_fe, y_train)


final_xgb_probs = final_xgb_pipeline.predict_proba(X_test_fe)[:, 1]


evaluate_model("Final XGBoost + Domain Feature Engineering", y_test, final_xgb_probs)

Final XGBoost + Domain Feature Engineering Performance 
ROC-AUC Score: 0.6769
PR-AUC Score:  0.2340
------------------------------


In [ ]:
from sklearn.metrics import classification_report, precision_recall_curve


precisions, recalls, thresholds = precision_recall_curve(y_test, final_xgb_probs)


target_recall = 0.65
idx = np.argmin(np.abs(recalls - target_recall))
opt_threshold = thresholds[idx]

print(f"Optimal Decision Threshold: {opt_threshold:.4f}\n")

y_pred_final = (final_xgb_probs >= opt_threshold).astype(int)

print("=== Final Pipeline Classification Report (Optimized Threshold) ===")
print(classification_report(y_test, y_pred_final, target_names=['Safe/Late (0)', 'High Risk (1)']))

Optimal Decision Threshold: 0.4614

=== Final Pipeline Classification Report (Optimized Threshold) ===
               precision    recall  f1-score   support

Safe/Late (0)       0.93      0.60      0.73     17606
High Risk (1)       0.17      0.65      0.27      2263

     accuracy                           0.61     19869
    macro avg       0.55      0.63      0.50     19869
 weighted avg       0.84      0.61      0.68     19869



In [ ]:

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols_fe),
        ('cat', cat_pipeline, cat_cols)
    ]
)


best_params = {k.replace('classifier__', ''): v for k, v in search.best_params_.items()}


final_xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier', XGBClassifier(**best_params, random_state=42, n_jobs=-1))
])


final_xgb_pipeline.fit(X_train_fe, y_train)

final_xgb_probs = final_xgb_pipeline.predict_proba(X_test_fe)[:, 1]


evaluate_model("Final XGBoost + Domain Feature Engineering", y_test, final_xgb_probs)

Final XGBoost + Domain Feature Engineering Performance 
ROC-AUC Score: 0.6769
PR-AUC Score:  0.2340
------------------------------


In [72]:

preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols_fe),
        ('cat', cat_pipeline, cat_cols)
    ]
)


best_params = {k.replace('classifier__', ''): v for k, v in search.best_params_.items()}


final_xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier', XGBClassifier(**best_params, random_state=42, n_jobs=-1))
])


final_xgb_pipeline.fit(X_train_fe, y_train)


final_xgb_probs = final_xgb_pipeline.predict_proba(X_test_fe)[:, 1]


evaluate_model("Final XGBoost + Domain Feature Engineering", y_test, final_xgb_probs)

Final XGBoost + Domain Feature Engineering Performance 
ROC-AUC Score: 0.6769
PR-AUC Score:  0.2340
------------------------------


In [73]:
import joblib
import os


artifacts_dir = '../app/artifacts'
os.makedirs(artifacts_dir, exist_ok=True)


joblib.dump(final_xgb_pipeline, os.path.join(artifacts_dir, 'diabetic_readmission_pipeline.pkl'))
config = {
    'optimal_threshold': opt_threshold,
    'feature_names': list(X_train_fe.columns)
}
joblib.dump(config, os.path.join(artifacts_dir, 'pipeline_config.pkl'))

print("S a v e d      S u c e s s f u l l y ")

S a v e d      S u c e s s f u l l y 
